# Feature Engineering

## Train-Validation Split

**Splitting Strategy**

In production, fraud models always predict future transactions from past data. A random split would allow future transaction patterns to influence training — violating the temporal ordering of real deployment.

Time-based split used for development:
- Training: first 80% of transactions by TransactionDT (~146 days)
- Validation: last 20% of transactions by TransactionDT (~36 days)

For final model evaluation: 5-fold stratified CV with feature engineering applied inside each fold to prevent target leakage from encoders.

**Note:** All feature engineering transformers (target encoders, scalers) are fitted exclusively on the training portion of each split/fold and applied to the validation portion.

In [1]:
import numpy as np
import pandas as pd

In [2]:
train_transaction = pd.read_csv("data/train_transaction.csv")
train_identity = pd.read_csv("data/train_identity.csv")

In [3]:
# Merge the datasets
df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

In [4]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [5]:
val_size = int(len(df) * 0.20)   # 20% validation

train_set = df.sort_values(by='TransactionDT', ascending=False)[val_size:] 
val_set = df.sort_values(by='TransactionDT', ascending=False)[:val_size]   # Top 20% of transaction date values - latest transactions

# Shuffle the datasets within themselves to keep it random
train_set = train_set.sample(frac=1, random_state=42).reset_index(drop=True)
val_set = val_set.sample(frac=1, random_state=42).reset_index(drop=True)

In [6]:
train_set.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,3195847,0,4824009,57.950,W,8320,476.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3237254,0,5959576,524.950,W,1939,360.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3281636,0,7267048,22.295,C,16136,204.0,185.0,visa,138.0,...,firefox 58.0,NaN,NaN,NaN,F,F,T,F,desktop,Windows
3,3441794,0,11636962,57.950,W,12501,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3393183,0,10262133,20.000,S,4436,174.0,150.0,visa,226.0,...,chrome 65.0,24.0,1280x720,match_status:2,T,F,T,T,mobile,NaN


In [7]:
val_set.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,3567977,0,15516011,34.00,W,12695,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3514680,0,13885965,53.95,W,8528,215.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3571209,0,15620941,100.00,R,16659,170.0,150.0,visa,226.0,...,edge 17.0,24.0,1365x768,match_status:2,T,F,T,T,desktop,Windows
3,3571251,0,15621795,68.95,W,3326,265.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3533740,0,14425996,39.00,W,11080,126.0,150.0,visa,138.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print("Length of train set:", len(train_set))
print("Length of validation set:", len(val_set))

Length of train set: 472432
Length of validation set: 118108


In [9]:
print("Fraud Rate for the train set:")
print(train_set['isFraud'].value_counts(normalize=True) * 100)

print("Fraud Rate for the validation set:")
val_set['isFraud'].value_counts(normalize=True) * 100

Fraud Rate for the train set:
isFraud
0    96.486478
1     3.513522
Name: proportion, dtype: float64
Fraud Rate for the validation set:


isFraud
0    96.559082
1     3.440918
Name: proportion, dtype: float64

- It can be seen that the fraud rate in both the sets is almost equal to the baseline fraud rate of the entire set.
- Hence we don't need to go for separate stratification based on the target variable here, the time-based split seems to be working fine.

In [10]:
X_train = train_set.drop(['isFraud'], axis=1).copy()
y_train = train_set['isFraud']

X_val = val_set.drop(['isFraud'], axis=1).copy()
y_val = val_set['isFraud']

## Feature Engineering

We will organise transformations by type rather than by feature family. 
This mirrors how a production ML pipeline is structured and makes the 
logic of each transformation step easier to follow and audit.

**Order of transformations:**
1. **Transaction Amount** — log1p to compress right skew
2. **Temporal Features** — derive Hour, Day, Week from TransactionDT
3. **Missing Indicators** — for columns where missingness carries fraud signal
4. **addr2** — binary flag for dominant region
5. **Target Encoding** — for high-cardinality categorical features
6. **One-Hot Encoding** — for low/moderate cardinality categorical features
7. **C Features** — log1p transformation
8. **V Feature Selection** — greedy correlation-based dimensionality reduction
9. **Velocity Aggregation Features** — card1/card2 groupby features

**Important:** All encoders and aggregations are fitted on X_train only 
and applied to both X_train and X_val. This prevents data leakage from 
the validation set influencing any learned transformation.

### 1. Transaction Amount

As the transaction values vary from 0 to 31K, we will take the log of the transaction amount column to compress this data and stabilize variance.

In [12]:
# Log tranformation on the TransactionAmt 

X_train['LogTransactionAmt'] = np.log1p(X_train['TransactionAmt'])
X_val['LogTransactionAmt'] = np.log1p(X_val['TransactionAmt'])

X_train.drop(['TransactionAmt'], axis=1, inplace=True)
X_val.drop(['TransactionAmt'], axis=1, inplace=True)

### 2. Temporal Features

We will use the TransactionDT column to derive the Transaction Hour, Day, and Week to help the model identify better patterns.

In [13]:
X_train['TransactionHour'] = (
    X_train['TransactionDT'] // 3600
) % 24

X_train['TransactionDay'] = (
    X_train['TransactionDT'] // 86400
)

X_train['TransactionWeek'] = (
    X_train['TransactionDT'] // (86400 * 7)
)

X_val['TransactionHour'] = (
    X_val['TransactionDT'] // 3600
) % 24

X_val['TransactionDay'] = (
    X_val['TransactionDT'] // 86400
)

X_val['TransactionWeek'] = (
    X_val['TransactionDT'] // (86400 * 7)
)

X_train.drop(['TransactionDT'], axis=1, inplace=True)
X_val.drop(['TransactionDT'], axis=1, inplace=True)

### 3. Missing Indicators

We will create missing indicators for those columns which have a high difference in the fraud rate when the value is missing v/s when it is present, showing us that missingness is informative. In our case, we're creating missing indicators for all columns which have a difference more than 0.01.

From EDA, we have chosen the following columns for missing indicator as per our criteria (difference > 0.01): D2, D3, D6-D15, dist1, dist2, id_01-id_11

In [14]:
cols = ['D2', 'D3', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', # Excluded: D1 (diff=0.001), D4 (diff=0.001), D5 (diff=0.007) — below 0.01 threshold
       'dist1', 'dist2',
       'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11']

for col in cols:
    X_train[f'{col}_missing'] = X_train[col].isnull().astype(int)
    X_val[f'{col}_missing'] = X_val[col].isnull().astype(int)

### 4. Addr2

We will create a is_dominant_region flag for this column as discussed during EDA.  
99% of the transactions has addr2=87 at a fraud rate of ~2.3%, whereas the rest of the transactions with different addr2 values have a fraud rate of ~11.7%.  
Hence, we will create a flag which tells us if the addr2 region is 87 or something else. 

In [15]:
X_train['is_dominant_region'] = (X_train['addr2'] == 87).astype(int)
X_val['is_dominant_region'] = (X_val['addr2'] == 87).astype(int)

### 5. Target Encoding

As discussed in the EDA, we will go for target encoding for the following columns: card1, card2, card3, card5, addr1, P/R_emaildomain

In [17]:
from category_encoders import TargetEncoder

te_cols = ['card1', 'card2', 'card3', 'card5', 'addr1', 'P_emaildomain', 'R_emaildomain']

te = TargetEncoder(cols = te_cols, smoothing=10)

X_train[te_cols] = te.fit_transform(X_train[te_cols], y_train)

X_val[te_cols] = te.transform(X_val[te_cols])

In [18]:
# Smoothing in target encoding:
# Without smoothing: a card1 value with 2 transactions, both fraud → 100% fraud rate
# This would massively overfit on rare categories
# 
# With smoothing=10: 
# encoded_value = (n * category_rate + smoothing * global_rate) / (n + smoothing)
# where n = number of times this category appears in train
#
# For rare categories (small n): pulls toward global fraud rate (3.5%)
# For common categories (large n): trusts the observed category rate
# smoothing=10 is a standard starting point

In [19]:
# Verify encoding worked correctly
print("Sample target encoded values for card1:")
print(X_train['card1'].describe())
print(f"\nAll values should be between 0 and 1 (fraud rates):")
print(f"Min: {X_train['card1'].min():.4f} | Max: {X_train['card1'].max():.4f}")

# Check val doesn't have NaN from unseen categories
print(f"\nNaN in val after encoding:")
print(X_val[te_cols].isnull().sum())

Sample target encoded values for card1:
count    472432.000000
mean          0.036017
std           0.053227
min           0.000000
25%           0.010127
50%           0.021277
75%           0.038878
max           0.919302
Name: card1, dtype: float64

All values should be between 0 and 1 (fraud rates):
Min: 0.0000 | Max: 0.9193

NaN in val after encoding:
card1            0
card2            0
card3            0
card5            0
addr1            0
P_emaildomain    0
R_emaildomain    0
dtype: int64
